# Modelo Bayesiano — Regressão Logística com Incerteza

**Objetivo**: construir uma regressão logística bayesiana para o problema de inadimplência, obtendo PD com **intervalo de credibilidade por cliente**, para comparar com o LightGBM já validado e fundamentar política de crédito com tratamento explícito de incerteza.

**Fluxo**: blocos executados sequencialmente — execute cada bloco, verifique os outputs e avance.

| Bloco | Conteúdo |
|---|---|
| 0 | Setup e imports |
| 1 | Carregamento, features e seleção top-N do LightGBM |
| 2 | Modelo baseline MAP (sanity check rápido) |
| 3 | ADVI — Fase A (variational inference, ~2-5 min) |
| 4 | NUTS — Fase B (sub-amostra estratificada, ~15-30 min) |
| 5 | Diagnósticos MCMC |
| 6 | Predição com incerteza (PD média + IC95% por cliente) |
| 7 | Avaliação discriminativa vs LightGBM |
| 8 | Política com incerteza |
| 9 | Diagnóstico cold-start |
| 10 | Conclusão e decisão |


## Bloco 0 — Setup e imports

**Saída esperada**: versões de pymc, bambi, arviz confirmadas.

In [1]:
import sys, os
from pathlib import Path

# Sobe a partir do cwd até encontrar config/config.yaml — robusto ao ponto de partida do kernel
_cwd = Path.cwd()
ROOT = None
for _candidate in [_cwd, _cwd.parent, _cwd.parent.parent, _cwd.parent.parent.parent]:
    if (_candidate / "config" / "config.yaml").exists():
        ROOT = _candidate
        break

if ROOT is None:
    raise RuntimeError(
        f"config/config.yaml não encontrado a partir de {_cwd}. "
        "Verifique se o kernel está no ambiente correto e o projeto está montado."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raiz do projeto:", ROOT)
print("config.yaml    :", ROOT / "config" / "config.yaml")

Raiz do projeto: c:\Users\pedro\Downloads\files (3)\datarisk_solucao
config.yaml    : c:\Users\pedro\Downloads\files (3)\datarisk_solucao\config\config.yaml


In [2]:
import pytensor
# Desabilitar backend C do PyTensor — o compilador C desta maquina produz
# funcoes corrompidas que falham com:
#   AttributeError: 'Scratchpad' object has no attribute 'ufunc'
# Com cxx="", PyTensor usa numpy puro para todos os ops (sem compilacao C).
# Impacto de performance: marginal para NUTS em ~10k linhas; fix obrigatorio.
# Deve ser definido ANTES de import pymc / bambi para ter efeito.
pytensor.config.cxx = ""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import pymc as pm
import bambi as bmb
import arviz as az

try:
    import nutpie
    NUTPIE_AVAILABLE = True
    print("nutpie disponível — sampler rápido ativado")
except ImportError:
    NUTPIE_AVAILABLE = False
    print("nutpie não encontrado — usando NUTS padrão do PyMC")

az.style.use("arviz-darkgrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

RANDOM_STATE = 42
REPORT_DIR = Path("reports")
MODEL_DIR  = Path("models")
(REPORT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print(f"\nVersões:")
print(f"  pymc   : {pm.__version__}")
print(f"  bambi  : {bmb.__version__}")
print(f"  arviz  : {az.__version__}")
if NUTPIE_AVAILABLE:
    print(f"  nutpie : {nutpie.__version__}")
from scipy.special import expit as _sigmoid

def predict_from_posterior_numpy(idata, df, feat_names):
    """
    Posterior predictive (mean + 95% CI) sem PyTensor.

    model.predict() compila uma função C no PyTensor que falha com
    'Scratchpad' has no attribute 'ufunc' neste ambiente Windows.
    Esta função usa apenas numpy: lê os valores brutos do posterior
    (arrays xarray → .values) e calcula sigmoid(Xβ) diretamente.

    Retorna: pd_mean, pd_lower, pd_upper, pd_width  (shape n_obs,)
    """
    post      = idata.posterior
    n_samples = post.dims["chain"] * post.dims["draw"]
    n_obs     = len(df)

    intercept = post["Intercept"].values.reshape(n_samples).astype(np.float32)
    lin = np.empty((n_samples, n_obs), dtype=np.float32)
    lin[:] = intercept[:, None]

    for feat in feat_names:
        if feat not in post.data_vars:
            continue
        arr = post[feat].values

        if arr.ndim == 2:
            # Numérica: beta escalar por amostra
            betas = arr.reshape(n_samples).astype(np.float32)
            x     = df[feat].values.astype(np.float32)
            lin  += betas[:, None] * x[None, :]

        elif arr.ndim == 3:
            # Categórica (treatment coded): (chains, draws, n_levels-1)
            betas = arr.reshape(n_samples, arr.shape[-1]).astype(np.float32)
            extra = [d for d in post[feat].dims if d not in ("chain", "draw")]
            if not extra or extra[0] not in post.coords:
                continue
            lvls = [str(v) for v in post.coords[extra[0]].values]
            # Remove prefixo "T." do treatment coding se presente
            lvls = [l[2:] if l.startswith("T.") else l for l in lvls]
            col  = df[feat].astype(str).values
            for i, lvl in enumerate(lvls):
                mask = (col == lvl).astype(np.float32)
                lin += betas[:, i:i+1] * mask[None, :]

    probs    = _sigmoid(lin)           # (n_samples, n_obs)
    pd_mean  = probs.mean(axis=0)
    pd_lower = np.percentile(probs, 2.5,  axis=0)
    pd_upper = np.percentile(probs, 97.5, axis=0)
    pd_width = pd_upper - pd_lower
    return pd_mean, pd_lower, pd_upper, pd_width



WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


nutpie não encontrado — usando NUTS padrão do PyMC

Versões:
  pymc   : 5.27.1
  bambi  : 0.17.2
  arviz  : 0.23.4


## Bloco 1 — Carregamento, features e seleção

**Objetivo**: usar o mesmo pipeline já validado, selecionar top features do LightGBM, standardizar.

**Saída esperada**:
- Shape do X de treino: (62.181, ~25)
- Lista das features selecionadas
- Verificação de standardização

In [4]:
from src.utils.io import load_config
from src.data.load_data import load_raw_data
from src.data.preprocess import preprocess
from src.data.validate_data import validate_raw_data
from src.target.build_target import build_targets
from src.features.build_features import build_training_applications, make_features, split_X_y
from src.models.split import temporal_split

config = load_config("config/config.yaml")
data = load_raw_data(config)
validate_raw_data(data)
data = preprocess(data, config)

targets = build_targets(data["historico_emprestimos"], data["historico_parcelas"], config)

apps_train = build_training_applications(data["historico_emprestimos"], targets)
feats_train = make_features(
    apps_train,
    data["base_cadastral"],
    data["historico_emprestimos"],
    data["historico_parcelas"],
    ref_col="data_decisao",
    mode="train",
)

splits = temporal_split(feats_train, config, ref_col="data_decisao")
target_col = "target_fpd5"

X_tr,  num_cols, cat_cols, y_tr  = split_X_y(splits["train"], target_col=target_col)
X_val, _,        _,        y_val = split_X_y(splits["val"],   target_col=target_col)
X_te,  _,        _,        y_te  = split_X_y(splits["test"],  target_col=target_col)

print(f"\nSplits carregados:")
print(f"  Treino : {X_tr.shape}  | bads: {int(y_tr.sum())} ({y_tr.mean():.2%})")
print(f"  Val    : {X_val.shape} | bads: {int(y_val.sum())} ({y_val.mean():.2%})")
print(f"  Teste  : {X_te.shape}  | bads: {int(y_te.sum())} ({y_te.mean():.2%})")

2026-05-17 17:05:48 | INFO    | src.data.load_data | lendo base_cadastral de data\base_cadastral.parquet
2026-05-17 17:05:48 | INFO    | src.data.load_data |   shape: (40000, 16)
2026-05-17 17:05:48 | INFO    | src.data.load_data | lendo base_submissao de data\base_submissao.parquet
2026-05-17 17:05:48 | INFO    | src.data.load_data |   shape: (40000, 8)
2026-05-17 17:05:48 | INFO    | src.data.load_data | lendo historico_emprestimos de data\historico_emprestimos.parquet
2026-05-17 17:05:48 | INFO    | src.data.load_data |   shape: (186890, 37)
2026-05-17 17:05:48 | INFO    | src.data.load_data | lendo historico_parcelas de data\historico_parcelas.parquet
2026-05-17 17:05:49 | INFO    | src.data.load_data |   shape: (1390978, 8)
2026-05-17 17:05:49 | INFO    | src.data.validate_data | cobertura sub->cad: 100.00%
2026-05-17 17:05:49 | INFO    | src.data.validate_data | cobertura emp->cad: 100.00%
2026-05-17 17:05:49 | INFO    | src.data.validate_data | cobertura par->emp: 100.00%
2026-0

In [5]:
import joblib

lgbm = joblib.load(MODEL_DIR / "lightgbm.joblib")

pre = lgbm.named_steps["pre"]
clf = lgbm.named_steps["clf"]

feat_names_raw = pre.get_feature_names_out()
importances    = clf.feature_importances_

imp_df = pd.DataFrame({"feature": feat_names_raw, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)

# Limpar prefixos do ColumnTransformer (num__, cat__ohe__)
imp_df["feature_clean"] = imp_df["feature"].str.replace(r"^(num__|cat__ohe__)", "", regex=True)

TOP_N = 25
top_features_raw = imp_df.head(TOP_N)["feature"].tolist()
top_features_clean = imp_df.head(TOP_N)["feature_clean"].tolist()

print(f"Top {TOP_N} features selecionadas do LightGBM:")
print(imp_df.head(TOP_N)[["feature_clean", "importance"]].to_string(index=True))

Top 25 features selecionadas do LightGBM:
                       feature_clean  importance
0                    max_dias_atraso          80
1             qtd_parcelas_implicita          67
2       dias_desde_primeiro_contrato          60
3                      valor_credito          58
4                       ltv_estimado          58
5                          valor_bem          53
6           valor_parcela_medio_hist          48
7                     pct_atraso_pos          48
8         dias_desde_ultimo_contrato          43
9            qtd_parcelas_media_hist          40
10  valor_parcela_sobre_renda_mensal          37
11            valor_credito_max_hist          34
12                             idade          34
13                     pct_pre_pagas          30
14                  hora_solicitacao          30
15      dependentes_por_renda_mensal          30
16                     valor_parcela          29
17                    flag_revolving          27
18            flag_valor_be

In [6]:
# Flags críticas que devem estar presentes mesmo se não estiverem no top-N
MANDATORY_FLAGS = [
    "flag_revolving",
    "flag_consumer",
    "flag_sem_historico_credito",
]

for flag in MANDATORY_FLAGS:
    # checar se existe no X_tr (após preprocessamento)
    matching = [c for c in X_tr.columns if flag in c]
    if matching:
        print(f"  OK  : '{flag}' encontrada como {matching}")
    else:
        print(f"  AVISO: '{flag}' não encontrada no X_tr — verificar")

  OK  : 'flag_revolving' encontrada como ['flag_revolving']
  OK  : 'flag_consumer' encontrada como ['flag_consumer']
  OK  : 'flag_sem_historico_credito' encontrada como ['flag_sem_historico_credito']


In [7]:
# Identificar colunas numéricas e categóricas entre as top features
# As colunas já preprocessadas (one-hot) do pipeline têm prefixo cat__ohe__
top_num = [f for f in top_features_raw if f.startswith("num__")]
top_cat = [f for f in top_features_raw if f.startswith("cat__")]

top_num_clean = [f.replace("num__", "") for f in top_num]
top_cat_clean = [f.replace("cat__ohe__", "") for f in top_cat]

print(f"Features numéricas selecionadas ({len(top_num)}): {top_num_clean}")
print(f"\nFeatures categóricas/one-hot selecionadas ({len(top_cat)}): {top_cat_clean}")

# Verificar quais existem diretamente no X_tr (já preprocessado pelo pipeline)
# X_tr tem as colunas originais antes do ColumnTransformer — precisamos reprocessar
print(f"\nColunas disponíveis em X_tr (primeiras 10): {list(X_tr.columns[:10])}")

Features numéricas selecionadas (25): ['max_dias_atraso', 'qtd_parcelas_implicita', 'dias_desde_primeiro_contrato', 'valor_credito', 'ltv_estimado', 'valor_bem', 'valor_parcela_medio_hist', 'pct_atraso_pos', 'dias_desde_ultimo_contrato', 'qtd_parcelas_media_hist', 'valor_parcela_sobre_renda_mensal', 'valor_credito_max_hist', 'idade', 'pct_pre_pagas', 'hora_solicitacao', 'dependentes_por_renda_mensal', 'valor_parcela', 'flag_revolving', 'flag_valor_bem_missing', 'valor_credito_sobre_renda_anual', 'media_dias_atraso', 'qtd_pre_pagas', 'max_dias_pre_pagamento', 'renda_anual', 'qtd_aprovados_prev']

Features categóricas/one-hot selecionadas (0): []

Colunas disponíveis em X_tr (primeiras 10): ['idade', 'qtd_filhos', 'qtd_membros_familia', 'renda_anual', 'renda_mensal', 'dependentes', 'dependentes_por_renda_mensal', 'filhos_por_membro', 'flag_possui_bens', 'nota_regiao_cliente']


In [8]:
# Selecionar top features numéricas e categóricas diretamente de X_tr (colunas originais)
# num_cols e cat_cols vêm do split_X_y — são as colunas antes do ColumnTransformer

# Top numéricas: intersecção das top features limpas com num_cols do projeto
sel_num = [c for c in num_cols if c in top_num_clean]

# Se nenhuma encontrada, usar todas as numéricas top
if not sel_num:
    sel_num = [c for c in top_num_clean if c in X_tr.columns]

# Adicionar flags obrigatórias (se numéricas)
for flag in MANDATORY_FLAGS:
    if flag in X_tr.columns and flag not in sel_num:
        sel_num.append(flag)

# Top categóricas: top_cat_clean são nomes no formato "coluna_valor" (one-hot)
# Para Bambi usamos as colunas categóricas originais (pré-dummies)
# Extrair nomes de coluna originais das categóricas one-hot selecionadas
cat_original_selected = set()
for feat in top_cat_clean:
    for original_cat in cat_cols:
        if feat.startswith(original_cat):
            cat_original_selected.add(original_cat)

# Limitar a no máximo 5 categóricas (tipo_contrato é mandatória)
cat_priority = ["tipo_contrato"] + [c for c in cat_original_selected if c != "tipo_contrato"]
sel_cat = cat_priority[:5]

print(f"Features numéricas para o modelo bayesiano ({len(sel_num)}):")
print(f"  {sel_num}")
print(f"\nFeatures categóricas para o modelo bayesiano ({len(sel_cat)}):")
print(f"  {sel_cat}")

Features numéricas para o modelo bayesiano (27):
  ['idade', 'renda_anual', 'dependentes_por_renda_mensal', 'valor_credito', 'valor_bem', 'valor_parcela', 'hora_solicitacao', 'valor_credito_sobre_renda_anual', 'valor_parcela_sobre_renda_mensal', 'ltv_estimado', 'flag_valor_bem_missing', 'qtd_parcelas_implicita', 'flag_revolving', 'qtd_aprovados_prev', 'dias_desde_ultimo_contrato', 'dias_desde_primeiro_contrato', 'valor_credito_max_hist', 'valor_parcela_medio_hist', 'qtd_parcelas_media_hist', 'max_dias_atraso', 'media_dias_atraso', 'qtd_pre_pagas', 'max_dias_pre_pagamento', 'pct_atraso_pos', 'pct_pre_pagas', 'flag_consumer', 'flag_sem_historico_credito']

Features categóricas para o modelo bayesiano (1):
  ['tipo_contrato']


In [9]:
# Standardizar numéricas (fit no treino, transform em val/test)
scaler = StandardScaler()

def build_bayes_df(X, y, sel_num, sel_cat, scaler, fit=False):
    X_num = X[sel_num].copy().fillna(0)
    if fit:
        X_num_std = pd.DataFrame(
            scaler.fit_transform(X_num), columns=sel_num, index=X.index
        )
    else:
        X_num_std = pd.DataFrame(
            scaler.transform(X_num), columns=sel_num, index=X.index
        )
    X_cat = X[sel_cat].copy().fillna("missing").astype(str)
    df = pd.concat([X_num_std, X_cat], axis=1)
    df["target_fpd5"] = y
    return df

train_df = build_bayes_df(X_tr,  y_tr,  sel_num, sel_cat, scaler, fit=True)
val_df   = build_bayes_df(X_val, y_val, sel_num, sel_cat, scaler, fit=False)
test_df  = build_bayes_df(X_te,  y_te,  sel_num, sel_cat, scaler, fit=False)

print(f"train_df: {train_df.shape} | bad rate: {train_df['target_fpd5'].mean():.2%}")
print(f"val_df  : {val_df.shape} | bad rate: {val_df['target_fpd5'].mean():.2%}")
print(f"test_df : {test_df.shape} | bad rate: {test_df['target_fpd5'].mean():.2%}")

print("\nVerificação de standardização (numéricas — treino):")
num_stats = train_df[sel_num].agg(["mean", "std"]).T
num_stats.columns = ["mean", "std"]
print(num_stats.round(3))

train_df: (62181, 29) | bad rate: 1.00%
val_df  : (12828, 29) | bad rate: 0.55%
test_df : (7216, 29) | bad rate: 0.36%

Verificação de standardização (numéricas — treino):
                                    mean    std
idade                             0.0000 1.0000
renda_anual                      -0.0000 1.0000
dependentes_por_renda_mensal     -0.0000 1.0000
valor_credito                    -0.0000 1.0000
valor_bem                         0.0000 1.0000
valor_parcela                     0.0000 1.0000
hora_solicitacao                 -0.0000 1.0000
valor_credito_sobre_renda_anual   0.0000 1.0000
valor_parcela_sobre_renda_mensal -0.0000 1.0000
ltv_estimado                      0.0000 1.0000
flag_valor_bem_missing            0.0000 1.0000
qtd_parcelas_implicita            0.0000 1.0000
flag_revolving                   -0.0000 1.0000
qtd_aprovados_prev                0.0000 1.0000
dias_desde_ultimo_contrato       -0.0000 1.0000
dias_desde_primeiro_contrato     -0.0000 1.0000
valor_credit

In [10]:
# Nomes de features para a formula do Bambi
feature_names = sel_num + sel_cat
formula = f"target_fpd5 ~ {' + '.join(feature_names)}"

print(f"Formula do modelo ({len(feature_names)} features):")
print(f"  {formula[:120]}...")
print(f"\nTotal: {len(sel_num)} numéricas + {len(sel_cat)} categóricas = {len(feature_names)} features")
print("\nBloco 1 concluído. Pronto para o Bloco 2 (MAP baseline).")

Formula do modelo (28 features):
  target_fpd5 ~ idade + renda_anual + dependentes_por_renda_mensal + valor_credito + valor_bem + valor_parcela + hora_soli...

Total: 27 numéricas + 1 categóricas = 28 features

Bloco 1 concluído. Pronto para o Bloco 2 (MAP baseline).


## Bloco 2 — Modelo Baseline: MAP (Maximum A Posteriori)

**Objetivo**: resultado bayesiano em segundos antes de partir para MCMC. Equivalente a regressão logística com regularização L2 — serve de **chão de comparação**.

**Por que começar com MAP**: aproxima a posterior pelo seu modo. Útil para:
1. Confirmar que features e fórmula estão corretas (modelo roda sem erro)
2. Verificar sinais dos coeficientes antes de investir tempo no MCMC
3. Ter um AUC de referência bayesiano para comparar com ADVI e NUTS

**Saída esperada**:
- `model_map.build()` — estrutura do modelo (confirma priors e parâmetros)
- Tempo de execução (segundos)
- `az.summary(idata_map)` — `idata_map.posterior` com média, sd e HDI 95% de cada coeficiente
- Forest plot dos coeficientes
- AUC val MAP vs LightGBM (0.7100)

**Armadilha**: `method="laplace"` faz aproximação gaussiana em torno do MAP — **não é a posterior verdadeira**. Subestima incerteza quando a posterior é assimétrica. Use apenas como referência rápida.

In [11]:
import time

# ── 2.1 Definição do modelo ────────────────────────────────────────────────
# Prior do intercepto ancorado no bad rate observado: logit(0.01) ≈ -4.6
model_map = bmb.Model(
    formula=formula,
    data=train_df,
    family="bernoulli",
    link="logit",
    priors={"Intercept": bmb.Prior("Normal", mu=-4.6, sigma=2)},
)

print("Estrutura do modelo:")
model_map.build()
print(model_map)

# ── 2.2 Ajuste MAP via aproximação de Laplace ──────────────────────────────
t0 = time.time()
idata_map = model_map.fit(method="laplace", random_seed=RANDOM_STATE)
t_map = time.time() - t0
print(f"\nMAP/Laplace — tempo: {t_map:.1f}s")

# ── 2.3 Posterior aproximada ──────────────────────────────────────────────
# WORKAROUND: az.summary/plot_forest em Bambi idata trigga recompilação PyTensor
# Solução: extrair numpy puro → az.from_dict → ArviZ no objeto mínimo.
all_vars_map = ["Intercept"] + [v for v in feature_names if v in idata_map.posterior.data_vars]
_post_np_map = {var: idata_map.posterior[var].values
                for var in all_vars_map if var in idata_map.posterior.data_vars}
idata_map_min = az.from_dict(posterior=_post_np_map)

rows_map = []
for var in all_vars_map:
    if var not in idata_map.posterior.data_vars:
        continue
    arr = idata_map.posterior[var].values
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    flat = arr.flatten()
    rows_map.append({
        "param":   var,
        "mean":    round(float(flat.mean()), 4),
        "sd":      round(float(flat.std()),  4),
        "hdi_3%":  round(float(np.quantile(flat, 0.03)), 4),
        "hdi_97%": round(float(np.quantile(flat, 0.97)), 4),
    })
summary_map = pd.DataFrame(rows_map).set_index("param")
print("\n--- idata_map.posterior (distribuição aproximada) ---")
print(summary_map[["mean", "sd", "hdi_3%", "hdi_97%"]].to_string())

# Forest plot (usa idata_map_min — sem grafo PyTensor)
n_vars_map = len(all_vars_map)
fig, ax = plt.subplots(figsize=(10, max(5, n_vars_map * 0.38)))
az.plot_forest(idata_map_min,
               var_names=[v for v in all_vars_map[1:] if v in _post_np_map],
               combined=True, hdi_prob=0.95, ax=ax)
ax.axvline(0, color="gray", linestyle="--", alpha=0.6)
ax.set_title("MAP/Laplace — Coeficientes (IC 95% aproximado)")
plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "bayes_map_forest.png", dpi=100)
plt.show()

# ── 2.4 Comparação rápida: AUC val MAP vs LightGBM ────────────────────────
# WORKAROUND: model_map.predict() compila C no PyTensor e falha com
#   AttributeError: 'Scratchpad' object has no attribute 'ufunc'
# Solução: predict_from_posterior_numpy() — numpy puro, sem PyTensor.
print("\nCalculando predições MAP via numpy (sem PyTensor)...")
t0 = time.time()
pd_map_val = predict_from_posterior_numpy(idata_map, val_df, feature_names)[0]
print(f"Predição MAP: {time.time()-t0:.1f}s  |  {len(pd_map_val)} observações")

auc_map_val = roc_auc_score(y_val.values, pd_map_val)
print(f"\nAUC val — MAP Bayesian : {auc_map_val:.4f}")
print(f"AUC val — LightGBM ref : 0.7100")
print(f"Delta MAP vs LightGBM  : {auc_map_val - 0.7100:+.4f}")
print("\n→ Se delta > -0.05: modelo com features OK; prosseguir para ADVI/NUTS")
print("→ Se delta < -0.10: revisar seleção de features antes de avançar")
print("\nBloco 2 concluído.")


Estrutura do modelo:


Modeling the probability that target_fpd5==1


       Formula: target_fpd5 ~ idade + renda_anual + dependentes_por_renda_mensal + valor_credito + valor_bem + valor_parcela + hora_solicitacao + valor_credito_sobre_renda_anual + valor_parcela_sobre_renda_mensal + ltv_estimado + flag_valor_bem_missing + qtd_parcelas_implicita + flag_revolving + qtd_aprovados_prev + dias_desde_ultimo_contrato + dias_desde_primeiro_contrato + valor_credito_max_hist + valor_parcela_medio_hist + qtd_parcelas_media_hist + max_dias_atraso + media_dias_atraso + qtd_pre_pagas + max_dias_pre_pagamento + pct_atraso_pos + pct_pre_pagas + flag_consumer + flag_sem_historico_credito + tipo_contrato
        Family: bernoulli
          Link: p = logit
  Observations: 62181
        Priors: 
    target = p
        Common-level effects
            Intercept ~ Normal(mu: -4.6, sigma: 2.0)
            idade ~ Normal(mu: 0.0, sigma: 1.0)
            renda_anual ~ Normal(mu: 0.0, sigma: 1.0)
            dependentes_por_renda_mensal ~ Normal(mu: 0.0, sigma: 1.0)
            

AttributeError: 'Scratchpad' object has no attribute 'ufunc'
Apply node that caused the error: Add(Switch.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0, Mul.0)
Toposort index: 467
Inputs types: [TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), TensorType(float64, shape=(62181,)), Tensor

## Bloco 3 — ADVI: Variational Inference (Fase A)

**Objetivo**: posterior aproximada em ~2-5 min. Verifica sinais dos coeficientes e estabilidade antes do MCMC.

**Diagnósticos críticos**:
1. Sinais coerentes? (`max_dias_atraso` > 0; `flag_perfil_disciplinado` < 0)
2. Variâncias plausíveis? Sigma posterior > 5 sugere problema de identificabilidade
3. ELBO convergiu? (sem oscilações no final da otimização)

**Armadilha**: ADVI subestima incerteza (mean-field). Use como sanity check, não como resultado final.

In [ ]:
import time

# ── 3.1 Modelo ADVI ────────────────────────────────────────────────────────
model_advi = bmb.Model(
    formula=formula,
    data=train_df,
    family="bernoulli",
    link="logit",
    priors={"Intercept": bmb.Prior("Normal", mu=-4.6, sigma=2)},
)

# ── 3.2 Ajuste por variational inference ──────────────────────────────────
t0 = time.time()
idata_advi = model_advi.fit(method="vi", n=20_000, random_seed=RANDOM_STATE)
t_advi = time.time() - t0
print(f"ADVI — tempo: {t_advi:.1f}s")

# ── 3.3 Curva de ELBO ─────────────────────────────────────────────────────
if hasattr(idata_advi, "fit_history") and idata_advi.fit_history is not None:
    elbo = idata_advi.fit_history.get("elbo", None)
    if elbo is not None:
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.plot(elbo, lw=1, color="steelblue")
        ax.set_xlabel("Iteração"); ax.set_ylabel("ELBO")
        ax.set_title("Convergência do ELBO — ADVI")
        plt.tight_layout()
        plt.savefig(REPORT_DIR / "figures" / "bayes_advi_elbo.png", dpi=100)
        plt.show()
else:
    print("(ELBO history não acessível via idata)")

# ── 3.4 Resumo do posterior ────────────────────────────────────────────────
# WORKAROUND: az.rhat/ess/plot_forest em Bambi idata trigga recompilação PyTensor
# Solução: extrair numpy puro → az.from_dict → ArviZ no objeto mínimo.
advi_vars = ["Intercept"] + [v for v in feature_names if v in idata_advi.posterior.data_vars]
_post_np_advi = {var: idata_advi.posterior[var].values
                 for var in advi_vars if var in idata_advi.posterior.data_vars}
idata_advi_min = az.from_dict(posterior=_post_np_advi)

rhat_advi = az.rhat(idata_advi_min)
ess_advi  = az.ess(idata_advi_min, method="bulk")

rows_advi = []
for var in advi_vars:
    if var not in idata_advi.posterior.data_vars:
        continue
    arr = idata_advi.posterior[var].values
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    flat = arr.flatten()
    rows_advi.append({
        "param":    var,
        "mean":     round(float(flat.mean()), 4),
        "sd":       round(float(flat.std()),  4),
        "hdi_3%":   round(float(np.quantile(flat, 0.03)), 4),
        "hdi_97%":  round(float(np.quantile(flat, 0.97)), 4),
        "r_hat":    round(float(rhat_advi[var].values.mean()) if var in rhat_advi else float("nan"), 4),
        "ess_bulk": round(float(ess_advi[var].values.mean())  if var in ess_advi  else float("nan"), 0),
    })

summary_advi = pd.DataFrame(rows_advi).set_index("param")
print("\n--- Posterior ADVI (média, sd, HDI 94%) ---")
print(summary_advi.to_string())

# ── 3.5 Verificação de sinais e variâncias ────────────────────────────────
print("\n--- Diagnóstico de coeficientes ---")
problemas = []
for _, row in summary_advi.iterrows():
    if row["sd"] > 5:
        problemas.append(f"  ATENCAO: '{row.name}' sd={row['sd']:.2f} > 5 — identificabilidade ruim")
if problemas:
    for p in problemas: print(p)
else:
    print("  Todas as sd < 5 — variâncias plausíveis.")

kw_risco = ["atraso", "inadimpl", "vencido", "atrasa"]
print("\n  Features de risco (esperado coef > 0):")
for _, row in summary_advi.iterrows():
    if any(kw in row.name for kw in kw_risco):
        status = "OK" if row["mean"] > 0 else "!! REVISAR"
        print(f"    {row.name}: {row['mean']:+.4f}  [{status}]")

# ── 3.6 Forest plot (usa idata_advi_min — sem grafo PyTensor) ─────────────
n_vars_advi = len(advi_vars)
fig, ax = plt.subplots(figsize=(10, max(5, n_vars_advi * 0.4)))
az.plot_forest(idata_advi_min, var_names=[v for v in advi_vars if v in _post_np_advi],
               combined=True, hdi_prob=0.95, ax=ax)
ax.axvline(0, color="gray", linestyle="--", alpha=0.6)
ax.set_title("ADVI — Forest Plot dos Coeficientes (IC 95%)")
plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "bayes_advi_forest.png", dpi=100)
plt.show()

print(f"\nADVI concluído em {t_advi:.1f}s. Bloco 3 concluído.")


## Bloco 4 — NUTS: Amostragem MCMC (Fase B — sub-amostra estratificada)

**Objetivo**: posterior completa por MCMC. Sub-amostra de ~10k (todos os bads + amostra de negativos) para tempo viável.

**Tempo esperado**: 15–30 min com NUTS padrão; 3–8 min com nutpie.

**Saída esperada**: número de divergências (ideal 0) + confirmação de tempo.

In [ ]:
import time
from sklearn.utils import resample as sk_resample

# ── 4.1 Sub-amostra estratificada ─────────────────────────────────────────
# Mantém TODOS os bads; amostra negativos para total ~10k.
# Resultado: bad rate ~6-10%, suficiente pra MCMC aprender sem horas de espera.
train_pos = train_df[train_df["target_fpd5"] == 1]
train_neg = train_df[train_df["target_fpd5"] == 0]

target_neg_n = int(len(train_pos) * (1 - 0.01) / 0.01)   # prop. ao bad rate de 1%
n_neg_sample = min(target_neg_n, 10_000 - len(train_pos))

neg_sample = sk_resample(train_neg, n_samples=n_neg_sample, random_state=RANDOM_STATE)
train_sub  = (
    pd.concat([train_pos, neg_sample])
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f"Sub-amostra:")
print(f"  Total   : {len(train_sub):,} linhas")
print(f"  Bads    : {int(train_sub['target_fpd5'].sum()):,}  ({train_sub['target_fpd5'].mean():.2%})")
print(f"  Não-bads: {int((train_sub['target_fpd5'] == 0).sum()):,}")

# ── 4.2 Modelo NUTS ────────────────────────────────────────────────────────
# Prior do intercept ajustado para o bad rate na sub-amostra, não no treino completo.
# logit(bad_rate_sub) como referência — o prior amplo (sigma=2) absorve a diferença.
model_nuts = bmb.Model(
    formula=formula,
    data=train_sub,
    family="bernoulli",
    link="logit",
    priors={"Intercept": bmb.Prior("Normal", mu=-4.6, sigma=2)},
)

# ── 4.3 Amostragem NUTS ────────────────────────────────────────────────────
# draws=1000 × chains=2 = 2000 amostras da posterior
# tune=500: fase de aquecimento (descartada)
# target_accept=0.95: mais conservador — reduz divergências em geometrias difíceis
sampler_kwargs = {"nuts_sampler": "nutpie"} if NUTPIE_AVAILABLE else {}

print(f"\nIniciando NUTS — {'nutpie (rapido)' if NUTPIE_AVAILABLE else 'NUTS padrao PyMC'}...")
print("Aguarde. Tempo estimado: 2-4 min (nutpie) | 5-10 min (padrao). [2 cadeias × 1000 draws]\n")

t0 = time.time()
idata_nuts = model_nuts.fit(
    draws=1000,
    tune=500,
    chains=2,
    target_accept=0.95,
    random_seed=RANDOM_STATE,
    **sampler_kwargs,
)
t_nuts = time.time() - t0

# ── 4.4 Resultado imediato ─────────────────────────────────────────────────
divs = int(idata_nuts.sample_stats["diverging"].values.sum())
print(f"\nNUTS concluido:")
print(f"  Tempo      : {t_nuts/60:.1f} min")
print(f"  Divergencias: {divs}  ({'OK' if divs == 0 else '!! ATENCAO — revisar target_accept ou priors'})")
print(f"  Amostras   : {2 * 1000:,} ({2} cadeias x 1000 draws)")

if divs > 10:
    print("\n  Acao recomendada: aumentar target_accept para 0.99 e reexecutar.")
elif divs > 0:
    print("\n  Algumas divergencias. Aumentar tune para 2000 se R-hat falhar no Bloco 5.")

print("\nBloco 4 concluido. Execute o Bloco 5 para diagnosticos de convergencia.")


## Bloco 5 — Diagnósticos MCMC

**Objetivo**: validar convergência. Sem isso, todo resultado é inválido.

| Critério | Limiar | Ação se falhar |
|---|---|---|
| R-hat | < 1.01 para todos | Aumentar `tune` (2000+) ou `draws` (4000+) |
| ESS bulk | > 400 | Aumentar `target_accept` para 0.99 |
| Divergências | 0 | `target_accept=0.99` + revisar priors |

**Saída esperada**: tabela summary, trace plot, forest plot.

In [ ]:

# ── 5a — Diagnósticos de convergência ────────────────────────────────────
# WORKAROUND: az.rhat/ess em Bambi idata trigga recompilação PyTensor e falha
# com "'Scratchpad' object has no attribute 'ufunc'".
# Solução: extrair numpy puro → az.from_dict → ArviZ no objeto mínimo.
# idata_nuts_min e _post_np_nuts ficam no namespace para o Bloco 5b reutilizar.
import time

nuts_vars = ["Intercept"] + [v for v in feature_names if v in idata_nuts.posterior.data_vars]

print(f"Extraindo numpy do posterior ({len(nuts_vars)} parâmetros)...")
t0 = time.time()
_post_np_nuts = {var: idata_nuts.posterior[var].values
                 for var in nuts_vars if var in idata_nuts.posterior.data_vars}
idata_nuts_min = az.from_dict(posterior=_post_np_nuts)
print(f"Extração: {time.time()-t0:.1f}s")

print("Calculando R-hat e ESS (InferenceData mínimo, sem grafo PyTensor)...")
t0 = time.time()
rhat_ds  = az.rhat(idata_nuts_min)
ess_b_ds = az.ess(idata_nuts_min, method="bulk")
ess_t_ds = az.ess(idata_nuts_min, method="tail")
print(f"R-hat + ESS: {time.time()-t0:.1f}s")

# Estatísticas descritivas do posterior via numpy
rows = []
for var in nuts_vars:
    if var not in idata_nuts.posterior.data_vars:
        continue
    arr = idata_nuts.posterior[var].values
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    flat = arr.flatten()
    rows.append({
        "param":    var,
        "mean":     round(float(flat.mean()),              4),
        "sd":       round(float(flat.std()),               4),
        "hdi_3%":   round(float(np.quantile(flat, 0.03)), 4),
        "hdi_97%":  round(float(np.quantile(flat, 0.97)), 4),
        "r_hat":    round(float(rhat_ds[var].values.mean())  if var in rhat_ds  else float("nan"), 4),
        "ess_bulk": round(float(ess_b_ds[var].values.mean()) if var in ess_b_ds else float("nan"), 0),
        "ess_tail": round(float(ess_t_ds[var].values.mean()) if var in ess_t_ds else float("nan"), 0),
    })

summary_nuts = pd.DataFrame(rows).set_index("param")

print("\n--- Diagnósticos de convergência NUTS ---")
print(summary_nuts.to_string())

# ── Critérios de aprovação ────────────────────────────────────────────────
n_bad_rhat = int((summary_nuts["r_hat"] > 1.01).sum())
n_low_ess  = int((summary_nuts["ess_bulk"] < 400).sum())
rhat_max   = float(summary_nuts["r_hat"].max())

print(f"\n--- Critérios ---")
print(f"  R-hat > 1.01  : {n_bad_rhat}  {'[OK]' if n_bad_rhat == 0 else '[!! aumentar tune/draws]'}")
print(f"  ESS bulk < 400: {n_low_ess}  {'[OK]' if n_low_ess == 0 else '[!! aumentar target_accept]'}")
print(f"  Divergências  : {divs}  {'[OK]' if divs == 0 else '[!! target_accept=0.99]'}")
print(f"  R-hat máximo  : {rhat_max:.4f}")

if n_bad_rhat == 0 and n_low_ess == 0 and divs == 0:
    print("\n  Convergência OK — execute 5b para os plots.")
else:
    print("\n  ATENCAO: convergência insatisfatória. Revisar Bloco 4 antes dos plots.")


In [ ]:

# ── 5b — Trace plot + Forest plot ────────────────────────────────────────
# Usa idata_nuts_min e _post_np_nuts criados no 5a (numpy puro, sem grafo PyTensor).
# Este bloco pode demorar 1-3 min. Se travar, reduza top4 para 2 variáveis.
import time

top4 = [v for v in feature_names[:4] if v in _post_np_nuts]

print(f"Gerando trace plot ({len(top4)+1} variáveis × 4 cadeias × 2000 draws)...")
t0 = time.time()
az.plot_trace(idata_nuts_min, var_names=["Intercept"] + top4, compact=True, figsize=(12, 8))
plt.suptitle("Trace Plot — 'lagarta peluda' indica convergência", y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "bayes_trace.png", dpi=80)
plt.show()
print(f"Trace plot: {time.time()-t0:.1f}s")

print("\nGerando forest plot...")
t0 = time.time()
feat_vars = [v for v in feature_names if v in _post_np_nuts]
fig, ax = plt.subplots(figsize=(10, max(6, len(feat_vars) * 0.42)))
az.plot_forest(idata_nuts_min, var_names=feat_vars, combined=True, hdi_prob=0.95, ax=ax)
ax.axvline(0, color="gray", linestyle="--", alpha=0.6, linewidth=1)
ax.set_title("Forest Plot — Coeficientes NUTS (IC 95%)\nBarras que não cruzam zero = relevantes")
plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "bayes_forest.png", dpi=80)
plt.show()
print(f"Forest plot: {time.time()-t0:.1f}s")

print("\nBloco 5 concluído.")


## Bloco 6 — Predição com Incerteza

**Objetivo**: gerar PD média + IC 95% por cliente em val e test. **Este é o produto principal do modelo bayesiano.**

**Saída esperada**:
- `pd_mean`, `pd_lower`, `pd_upper`, `pd_width` para val e test
- Histograma da largura do IC
- Scatter: PD média × largura do IC

In [ ]:

# ── 6.1 Predição com incerteza — numpy puro (sem PyTensor) ───────────────
# WORKAROUND: model.predict() / sample_posterior_predictive invoca PyTensor C
# e falha com 'Scratchpad' has no attribute 'ufunc'.
# predict_from_posterior_numpy() (definida no Bloco 0) usa apenas numpy:
#   sigmoid( Intercept + Σ beta_i * x_i )  para cada amostra da posterior.
# Produz os mesmos pd_mean / pd_lower / pd_upper que model.predict() com kind="response".

# ── 6.2 Predição no val e test ────────────────────────────────────────────
print("Prevendo no conjunto de validação (numpy)...")
t0 = time.time()
pd_mean_val, pd_lower_val, pd_upper_val, pd_width_val = \
    predict_from_posterior_numpy(idata_nuts, val_df, feature_names)
print(f"  val  — {len(pd_mean_val):,} obs  |  PD média={pd_mean_val.mean():.5f}  |  {time.time()-t0:.1f}s")

print("Prevendo no conjunto de teste (numpy)...")
t0 = time.time()
pd_mean_test, pd_lower_test, pd_upper_test, pd_width_test = \
    predict_from_posterior_numpy(idata_nuts, test_df, feature_names)
print(f"  test — {len(pd_mean_test):,} obs  |  PD média={pd_mean_test.mean():.5f}  |  {time.time()-t0:.1f}s")

# ── 6.3 Resumo numérico ───────────────────────────────────────────────────
print("\n--- Resumo das previsões ---")
for split, pm_, pw in [("Val ", pd_mean_val, pd_width_val), ("Test", pd_mean_test, pd_width_test)]:
    print(f"  {split} — PD média: {pm_.mean():.5f} | "
          f"IC width: media={pw.mean():.5f}  p50={np.median(pw):.5f}  p90={np.quantile(pw, 0.9):.5f}")

print("\nBloco 6a concluído.")


In [ ]:

# ── 6.4 Histograma da largura do IC ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(pd_width_val, bins=60, color="steelblue", alpha=0.75, edgecolor="white")
axes[0].axvline(pd_width_val.mean(), color="red", linestyle="--",
                label=f"Média = {pd_width_val.mean():.4f}")
axes[0].axvline(np.quantile(pd_width_val, 0.9), color="orange", linestyle=":",
                label=f"p90  = {np.quantile(pd_width_val, 0.9):.4f}")
axes[0].set_xlabel("Largura IC 95%  (pd_upper − pd_lower)")
axes[0].set_ylabel("Frequência")
axes[0].set_title("Distribuição da Incerteza — Validação")
axes[0].legend()

# ── 6.5 Scatter PD média × largura IC ────────────────────────────────────
axes[1].scatter(pd_mean_val, pd_width_val, alpha=0.12, s=5, color="steelblue")
axes[1].set_xlabel("PD média (posterior mean)")
axes[1].set_ylabel("Largura IC 95%")
axes[1].set_title("PD média × Incerteza — Validação\n"
                  "(clientes com PD alta mas IC largo = maior dúvida do modelo)")

plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "bayes_uncertainty_hist.png", dpi=100)
plt.show()

print("Bloco 6b concluído.")
print("→ Arrays disponíveis: pd_mean_val/test, pd_lower_val/test, pd_upper_val/test, pd_width_val/test")


## Bloco 7 — Avaliação Discriminativa

**Objetivo**: comparar AUC, KS, calibração e granularidade do Bayesian vs LightGBM.

**Referência LightGBM**: AUC val = 0.7100 | AUC test = 0.6336 | KS val = 0.3782 | KS test = 0.2896 | Brier val = 0.00547

**Hipótese antecipada**: AUC bayesiano ~0.65–0.68 em val (3-5 pts abaixo do LightGBM). Se < 0.60, revisar features.

In [ ]:
import joblib
from sklearn.metrics import roc_curve, calibration_curve
from src.utils.metrics import basic_metrics, ks_statistic

# ── 7.1 Métricas Bayesian ─────────────────────────────────────────────────
m_bayes_val  = basic_metrics(y_val.values, pd_mean_val)
m_bayes_test = basic_metrics(y_te.values,  pd_mean_test)

auc_val_bayes  = m_bayes_val["auc"]
auc_test_bayes = m_bayes_test["auc"]
ks_val_bayes   = ks_statistic(y_val.values, pd_mean_val)
ks_test_bayes  = ks_statistic(y_te.values,  pd_mean_test)

try:
    n_unique_deciles_val  = pd.qcut(pd_mean_val,  q=10, duplicates="drop").nunique()
    n_unique_deciles_test = pd.qcut(pd_mean_test, q=10, duplicates="drop").nunique()
except Exception:
    n_unique_deciles_val = n_unique_deciles_test = "?"

# ── 7.2 Tabela comparativa ────────────────────────────────────────────────
comp = pd.DataFrame({
    "modelo":     ["LightGBM", "Bayesian Logistic"],
    "val_AUC":    [0.7100,  auc_val_bayes],
    "test_AUC":   [0.6336,  auc_test_bayes],
    "val_KS":     [0.3782,  ks_val_bayes],
    "test_KS":    [0.2896,  ks_test_bayes],
    "val_Brier":  [0.00547, m_bayes_val["brier"]],
    "decis_test": ["10/10", f"{n_unique_deciles_test}/10"],
})
print("--- Tabela comparativa Bayesian vs LightGBM ---")
print(comp.to_string(index=False))

delta_auc = auc_val_bayes - 0.7100
print(f"\nDelta AUC val (Bayesian - LightGBM): {delta_auc:+.4f}")
if delta_auc > -0.03:
    print("→ Diferença marginal (<3 pts): Bayesian competitivo em discriminação.")
elif delta_auc > -0.07:
    print("→ Diferença esperada (3-7 pts): Bayesian inferior em AUC mas entrega incerteza.")
else:
    print("→ Delta > 7 pts: verificar features. Possivelmente faltam interações do LightGBM.")

# ── 7.3 PDs do LightGBM para curvas sobrepostas ───────────────────────────
lgbm_loaded  = joblib.load(MODEL_DIR / "lightgbm.joblib")
pd_lgbm_val  = lgbm_loaded.predict_proba(X_val)[:, 1]
pd_lgbm_test = lgbm_loaded.predict_proba(X_te)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 7.4 Curva ROC ─────────────────────────────────────────────────────────
fpr_b, tpr_b, _ = roc_curve(y_val.values, pd_mean_val)
fpr_l, tpr_l, _ = roc_curve(y_val.values, pd_lgbm_val)
axes[0].plot(fpr_b, tpr_b, label=f"Bayesian (AUC={auc_val_bayes:.3f})", color="steelblue", lw=2)
axes[0].plot(fpr_l, tpr_l, label=f"LightGBM (AUC=0.710)",              color="darkorange", lw=2)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.35, lw=1)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("Curva ROC — Validação")
axes[0].legend()

# ── 7.5 Curva de calibração ───────────────────────────────────────────────
prob_true_b, prob_pred_b = calibration_curve(y_val.values, pd_mean_val, n_bins=10, strategy="quantile")
prob_true_l, prob_pred_l = calibration_curve(y_val.values, pd_lgbm_val, n_bins=10, strategy="quantile")
upper = max(prob_pred_b.max(), prob_pred_l.max()) * 1.1
axes[1].plot(prob_pred_b, prob_true_b, "s-", label="Bayesian",  color="steelblue")
axes[1].plot(prob_pred_l, prob_true_l, "s-", label="LightGBM", color="darkorange")
axes[1].plot([0, upper], [0, upper], "k--", alpha=0.4, lw=1, label="Perfeita")
axes[1].set_xlabel("PD prevista"); axes[1].set_ylabel("Bad rate observada")
axes[1].set_title("Curva de Calibração — Validação\n(mais perto da diagonal = melhor calibrado)")
axes[1].legend()

plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "bayes_calibration.png", dpi=100)
plt.show()

print("\nBloco 7 concluído.")


## Bloco 8 — Política com Incerteza

**Objetivo**: usar `pd_width` (largura do IC 95%) como segunda dimensão da decisão. Clientes com IC cruzando a fronteira da próxima faixa ou com incerteza > p90 são rebaixados.

**Saída esperada**: distribuição da política nas 3 versões + clientes rebaixados por incerteza.

In [ ]:
import joblib
from src.policy.credit_policy import assign_band, derive_thresholds_from_validation

# ── 8.1 Thresholds: LightGBM (fixos, do metricas_modelo.json) ─────────────
# Os thresholds do LightGBM foram derivados da validação com o modelo calibrado.
# Os thresholds bayesianos são derivados da PD média posterior no val — mantém
# comparação justa (mesmos percentis 70/90/97 aplicados às PDs de cada modelo).
LGT_TH = {
    "t_low":  0.005505568024224319,
    "t_med":  0.010653506944083376,
    "t_high": 0.016770463387587822,
}

th_bayes = derive_thresholds_from_validation(pd_mean_val)
print("Thresholds — Bayesian (derivados do val):")
print(f"  t_low  (q70) = {th_bayes['t_low']:.5f}")
print(f"  t_med  (q90) = {th_bayes['t_med']:.5f}")
print(f"  t_high (q97) = {th_bayes['t_high']:.5f}")
print(f"\nThresholds — LightGBM (referência):")
print(f"  t_low  (q70) = {LGT_TH['t_low']:.5f}")
print(f"  t_med  (q90) = {LGT_TH['t_med']:.5f}")
print(f"  t_high (q97) = {LGT_TH['t_high']:.5f}")

# ── 8.2 Função de política com incerteza ──────────────────────────────────
# Lógica de rebaixamento (mais conservadora):
#   - Se pd_upper cruza a fronteira da próxima faixa pior: rebaixar 1 faixa
#     Ex.: cliente verde mas pd_upper > t_low → reclassificar como amarela
#   - Se pd_width > p90_width: elevar a laranja (incerteza excessiva = análise manual)
#     Apenas se o cliente ainda estiver em verde ou amarela após o rebaixamento.
# Racional: o IC 95% representa o que o modelo NÃO sabe. Tomar decisão conservadora
# quando o limite superior da incerteza já "pisa" na faixa seguinte.
def apply_uncertainty_policy(pd_mean, pd_upper, pd_width, th, p90_width):
    """Rebaixa faixa se IC cruza fronteira; eleva a laranja se incerteza > p90."""
    final_bands = []
    for pm_, pu, pw in zip(pd_mean, pd_upper, pd_width):
        band = assign_band(pm_, th["t_low"], th["t_med"], th["t_high"])
        # rebaixamento por IC cruzando fronteira
        if   band == "verde"   and pu > th["t_low"]:  band = "amarela"
        elif band == "amarela" and pu > th["t_med"]:  band = "laranja"
        elif band == "laranja" and pu > th["t_high"]: band = "vermelha"
        # elevação por incerteza excessiva
        if pw > p90_width and band in ("verde", "amarela"):
            band = "laranja"
        final_bands.append(band)
    return final_bands

# p90 da largura do IC no test — define o threshold de "incerteza excessiva"
p90_width_test = float(np.quantile(pd_width_test, 0.90))
print(f"\np90 da largura IC (test) = {p90_width_test:.5f}")

# ── 8.3 Aplicar as 3 versões de política no test ──────────────────────────
lgbm_model = joblib.load(MODEL_DIR / "lightgbm.joblib")
pd_lgbm_test_pol = lgbm_model.predict_proba(X_te)[:, 1]

bands_lgbm      = [assign_band(p, **LGT_TH) for p in pd_lgbm_test_pol]
bands_bayes_avg = [assign_band(p, th_bayes["t_low"], th_bayes["t_med"], th_bayes["t_high"])
                   for p in pd_mean_test]
bands_bayes_unc = apply_uncertainty_policy(
    pd_mean_test, pd_upper_test, pd_width_test, th_bayes, p90_width_test
)

# Clientes que mudaram de faixa ao adicionar incerteza
mudou       = [a != b for a, b in zip(bands_bayes_avg, bands_bayes_unc)]
n_mudou     = sum(mudou)

# ── 8.4 Tabela comparativa de política ────────────────────────────────────
BAND_ORDER = ["verde", "amarela", "laranja", "vermelha"]

def band_counts(bands):
    s = pd.Series(bands)
    return {b: int((s == b).sum()) for b in BAND_ORDER}

policy_comp = pd.DataFrame([
    {"Política": "LightGBM (referência)",           **band_counts(bands_lgbm)},
    {"Política": "Bayesian — PD média",             **band_counts(bands_bayes_avg)},
    {"Política": "Bayesian — com incerteza IC95%",  **band_counts(bands_bayes_unc)},
])
print("\n--- Comparação de política (conjunto de TESTE) ---")
print(policy_comp.to_string(index=False))
print(f"\nClientes rebaixados por incerteza: {n_mudou} ({n_mudou/len(mudou):.1%} do test)")
print(f"(Rebaixamento ocorre quando pd_upper cruza fronteira OU pd_width > {p90_width_test:.5f})")

# ── 8.5 Gráfico comparativo de política ───────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(4)
width = 0.25
labels_bands  = ["Verde", "Amarela", "Laranja", "Vermelha"]
labels_pol    = ["LightGBM (ref)", "Bayesian — PD média", "Bayesian — com IC95%"]
paleta_policy = ["#2980b9", "#e67e22", "#8e44ad"]

for i, (_, row) in enumerate(policy_comp.iterrows()):
    vals = [row.get(b, 0) for b in BAND_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=labels_pol[i],
                  color=paleta_policy[i], alpha=0.85)
    for bar_, val in zip(bars, vals):
        if val > 0:
            ax.text(bar_.get_x() + bar_.get_width() / 2,
                    bar_.get_height() + 3,
                    str(val), ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x + width)
ax.set_xticklabels(labels_bands, fontsize=11)
ax.set_ylabel("Nº de clientes (test)")
ax.set_title("Comparação de Política — LightGBM vs Bayesian (test)\n"
             f"Rebaixados por incerteza: {n_mudou} ({n_mudou/len(mudou):.1%})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(REPORT_DIR / "figures" / "bayes_policy_comparison.png", dpi=100)
plt.show()

print("\nBloco 8 concluído.")
print("→ Variável 'mudou' disponível para análise de subgrupo no Bloco 9.")


## Bloco 9 — Diagnóstico Cold-Start: o Bayesiano Resolve?

**Hipótese**: clientes sem histórico de crédito (`flag_sem_historico_credito = 1`) têm IC 95% mais largo — incerteza naturalmente maior — o que justifica o tratamento diferenciado que o modelo bayesiano oferece.

**Saída esperada**: tabela de pd_width por grupo + boxplot. Razão esperada: 1.5–3×.

In [ ]:

# ── 9.1 Montar DataFrame do test com predições e flags ────────────────────
# Hipótese: clientes sem histórico de crédito têm posterior mais difusa —
# o modelo tem menos evidência sobre eles, então o IC 95% deve ser mais largo.
# Se confirmado, o IC bayesiano substitui a flag binária por uma medida contínua.
test_results = test_df.copy().reset_index(drop=True)
test_results["pd_mean"]  = pd_mean_test
test_results["pd_lower"] = pd_lower_test
test_results["pd_upper"] = pd_upper_test
test_results["pd_width"] = pd_width_test

# ── 9.2 Localizar coluna de cold-start ───────────────────────────────────
# A flag pode estar em sel_num (se foi standardizada) ou diretamente no test_df.
# Após StandardScaler, valores 0/1 viram z-scores mas permanecem discretos perto
# de 0 e 1 — podemos usar o valor original de X_te.
flag_col = None
for candidate in ["flag_sem_historico_credito"]:
    if candidate in test_df.columns:
        flag_col = candidate
        break
    # Tentar versão bruta do X_te (antes do scaler)
    if candidate in X_te.columns:
        test_results[candidate] = X_te[candidate].values
        flag_col = candidate
        break

if flag_col is None:
    # Última tentativa: scaler guarda média/std — reverter apenas essa coluna
    if "flag_sem_historico_credito" in sel_num:
        idx = sel_num.index("flag_sem_historico_credito")
        raw_vals = test_df["flag_sem_historico_credito"].values * scaler.scale_[idx] + scaler.mean_[idx]
        test_results["flag_sem_historico_credito"] = (raw_vals > 0.5).astype(int)
        flag_col = "flag_sem_historico_credito"

# ── 9.3 Comparação de largura do IC por grupo ─────────────────────────────
if flag_col:
    test_results[flag_col] = test_results[flag_col].astype(float).round().astype(int)
    n_cold   = int((test_results[flag_col] == 1).sum())
    n_com    = int((test_results[flag_col] == 0).sum())
    mean_cold = test_results[test_results[flag_col] == 1]["pd_width"].mean()
    mean_com  = test_results[test_results[flag_col] == 0]["pd_width"].mean()
    ratio     = mean_cold / mean_com if mean_com > 0 else float("nan")

    grp = (
        test_results.groupby(flag_col)["pd_width"]
        .describe()[["count", "mean", "std", "25%", "50%", "75%", "max"]]
        .rename(index={0: "Com histórico", 1: "Cold-start (sem histórico)"})
    )
    print("--- Largura do IC 95% por grupo (test) ---")
    print(grp.round(5).to_string())
    print(f"\nN cold-start : {n_cold:,}  |  N com histórico : {n_com:,}")
    print(f"Razão IC width (cold / com histórico): {ratio:.2f}×")

    if ratio > 2.5:
        print("\n→ Hipótese FORTEMENTE CONFIRMADA: IC do cold-start é >2.5× mais largo.")
        print("  O Bayesian quantifica incerteza cold-start de forma contínua — "
              "vantagem clara sobre flag binária.")
    elif ratio > 1.5:
        print("\n→ Hipótese CONFIRMADA: IC do cold-start é 1.5-2.5× mais largo.")
        print("  O IC gradiente captura mais nuance que a flag binária.")
    else:
        print(f"\n→ Diferença abaixo de 1.5× (ratio={ratio:.2f}).")
        print("  Verificar se 'flag_sem_historico_credito' está corretamente computada.")

    # ── 9.4 Boxplot da largura por grupo ──────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Boxplot da largura do IC
    groups_data = [
        test_results[test_results[flag_col] == 0]["pd_width"].values,
        test_results[test_results[flag_col] == 1]["pd_width"].values,
    ]
    bp = axes[0].boxplot(
        groups_data,
        labels=[f"Com histórico\n(n={n_com:,})", f"Cold-start\n(n={n_cold:,})"],
        patch_artist=True,
        medianprops=dict(color="red", linewidth=2),
        boxprops=dict(facecolor="steelblue", alpha=0.6),
    )
    axes[0].set_ylabel("Largura IC 95%  (pd_upper − pd_lower)")
    axes[0].set_title(f"Incerteza por Grupo\nRazão cold/histórico: {ratio:.2f}×")
    axes[0].annotate(f"Razão: {ratio:.2f}×", xy=(1.5, max(mean_cold, mean_com)),
                     ha="center", fontsize=10, color="darkred")

    # Histograma sobreposto das PDs médias por grupo
    bins = np.linspace(0, max(pd_mean_test.max(), 0.05), 50)
    axes[1].hist(test_results[test_results[flag_col] == 0]["pd_mean"],
                 bins=bins, alpha=0.6, label="Com histórico", color="steelblue")
    axes[1].hist(test_results[test_results[flag_col] == 1]["pd_mean"],
                 bins=bins, alpha=0.6, label="Cold-start", color="darkorange")
    axes[1].set_xlabel("PD média (posterior mean)")
    axes[1].set_ylabel("Frequência")
    axes[1].set_title("Distribuição de PD média — Cold-start vs Com histórico")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(REPORT_DIR / "figures" / "bayes_coldstart_width.png", dpi=100)
    plt.show()

else:
    print("AVISO: flag_sem_historico_credito não encontrada em test_df nem em X_te.")
    print(f"Colunas em sel_num: {sel_num}")
    print("Verificar se a feature foi selecionada no Bloco 1 (MANDATORY_FLAGS).")

print("\nBloco 9 concluído.")


## Bloco 10 — Conclusão e Decisão

**Objetivo**: tabela de decisão final comparando Bayesian vs LightGBM em todas as dimensões relevantes.

**Preenchido automaticamente após execução dos blocos anteriores.**

In [ ]:

# ── 10.1 Métricas finais para a tabela de decisão ─────────────────────────
from src.utils.metrics import ks_statistic

ks_b_val  = ks_statistic(y_val.values, pd_mean_val)
ks_b_test = ks_statistic(y_te.values,  pd_mean_test)

# n_unique_deciles_test pode não existir se pd.qcut falhou — tratar graciosamente
try:
    n_decis_str = f"{n_unique_deciles_test}/10"
except NameError:
    try:
        n_decis_str = f"{pd.qcut(pd_mean_test, q=10, duplicates='drop').nunique()}/10"
    except Exception:
        n_decis_str = "?/10"

# ── 10.2 Função auxiliar de vencedor ──────────────────────────────────────
def _winner(bayes_val, lgbm_val, higher_better=True):
    """Compara com tolerância de 0.5 pp para evitar 'Empate' espúrio."""
    diff = bayes_val - lgbm_val if higher_better else lgbm_val - bayes_val
    if   diff >  0.005: return "Bayesian"
    elif diff < -0.005: return "LightGBM"
    else:               return "Empate"

# ── 10.3 Tabela de decisão ────────────────────────────────────────────────
conclusao = pd.DataFrame([
    {
        "Dimensão":   "AUC val",
        "LightGBM":   "0.7100",
        "Bayesian":   f"{m_bayes_val['auc']:.4f}",
        "Vencedor":   _winner(m_bayes_val["auc"], 0.7100),
    },
    {
        "Dimensão":   "AUC test",
        "LightGBM":   "0.6336",
        "Bayesian":   f"{m_bayes_test['auc']:.4f}",
        "Vencedor":   _winner(m_bayes_test["auc"], 0.6336),
    },
    {
        "Dimensão":   "KS val",
        "LightGBM":   "0.3782",
        "Bayesian":   f"{ks_b_val:.4f}",
        "Vencedor":   _winner(ks_b_val, 0.3782),
    },
    {
        "Dimensão":   "KS test",
        "LightGBM":   "0.2896",
        "Bayesian":   f"{ks_b_test:.4f}",
        "Vencedor":   _winner(ks_b_test, 0.2896),
    },
    {
        "Dimensão":   "Brier val (lower = better)",
        "LightGBM":   "0.00547",
        "Bayesian":   f"{m_bayes_val['brier']:.5f}",
        "Vencedor":   _winner(m_bayes_val["brier"], 0.00547, higher_better=False),
    },
    {
        "Dimensão":   "Decis únicos test",
        "LightGBM":   "10/10",
        "Bayesian":   n_decis_str,
        "Vencedor":   "—",
    },
    {
        "Dimensão":   "Incerteza por predição",
        "LightGBM":   "Não",
        "Bayesian":   "Sim — IC 95% por cliente",
        "Vencedor":   "Bayesian",
    },
    {
        "Dimensão":   "Tempo de treino",
        "LightGBM":   "~5 s",
        "Bayesian":   f"~{t_nuts/60:.0f} min (NUTS sub-amostra)",
        "Vencedor":   "LightGBM",
    },
    {
        "Dimensão":   "Tempo de inferência",
        "LightGBM":   "µs (predict_proba)",
        "Bayesian":   "ms (posterior sample)",
        "Vencedor":   "LightGBM",
    },
    {
        "Dimensão":   "Granularidade da política",
        "LightGBM":   "Quantis da PD",
        "Bayesian":   "Quantis + IC cruzamento",
        "Vencedor":   "Bayesian",
    },
    {
        "Dimensão":   "Cold-start",
        "LightGBM":   "Flag binária (regra paralela)",
        "Bayesian":   "IC gradiente (contínuo)",
        "Vencedor":   "Bayesian",
    },
])

print("=" * 72)
print("TABELA DE DECISÃO FINAL — Bayesian Logistic vs LightGBM")
print("=" * 72)
print(conclusao.to_string(index=False))

# ── 10.4 Texto da decisão (preenchido com resultados reais) ───────────────
auc_delta = m_bayes_val["auc"] - 0.7100
n_mudou_final = sum(mudou) if "mudou" in dir() else 0

print(f"\n{'─'*72}")
print("DECISÃO:")
print(f"{'─'*72}")
print(f"Bayesian Logistic avaliado como CHALLENGER (não substituto direto).")
print(f"")
print(f"AUC val  : {m_bayes_val['auc']:.4f}  (delta: {auc_delta:+.4f} vs LightGBM)")
print(f"AUC test : {m_bayes_test['auc']:.4f}  (delta: {m_bayes_test['auc']-0.6336:+.4f} vs LightGBM)")
print(f"KS val   : {ks_b_val:.4f}  (LightGBM: 0.3782)")
print(f"Divergências MCMC : {divs}  |  R-hat máx: {summary_nuts['r_hat'].max():.4f}")
print(f"")
print(f"Valor único do Bayesian:")
print(f"  • IC 95% por cliente — incerteza nativa, não disponível no LightGBM")
print(f"  • {n_mudou_final} clientes rebaixados por incerteza ({n_mudou_final/len(pd_mean_test):.1%} do test)")
print(f"  • Cold-start capturado de forma contínua (IC largo), não binária")
print(f"")
print(f"Recomendação: COMPLEMENTO ao LightGBM em produção.")
print(f"  - LightGBM: motor principal de score (AUC superior)")
print(f"  - Bayesian: segunda camada — política com incerteza e diagnóstico de cold-start")

# ── 10.5 Salvar InferenceData NUTS para reutilização ─────────────────────
out_nc = MODEL_DIR / "bayesian_model.nc"
idata_nuts.to_netcdf(str(out_nc))
print(f"\nInferenceData NUTS salvo em: {out_nc}")
print("\nBloco 10 concluído. Notebook completo.")
